In [ ]:

import os
import datetime
import re
import math
from collections import OrderedDict
import multiprocessing
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow.keras.backend as K
import tensorflow.keras.layers as KL
import tensorflow.keras.utils as KU
from tensorflow.python.eager import context
import tensorflow.keras.models as KM

from mrcnn import utils
import sys
from mrcnn.parallel_model import ParallelModel

from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
config = ConfigProto() 
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)

# Requires TensorFlow 2.0+
from distutils.version import LooseVersion
assert LooseVersion(tf.__version__) >= LooseVersion("2.0")

tf.compat.v1.disable_eager_execution()

NotFoundError: dlopen(/Users/vanshikaarora/Desktop/Mask-rcnn-from-scratch-personal/venv/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib, 0x0006): Symbol not found: __ZN10tensorflow16TensorShapeProtoC1ERKS0_
  Referenced from: <10B7FC95-0B10-3E4E-84D0-79A2D52E4D78> /Users/vanshikaarora/Desktop/Mask-rcnn-from-scratch-personal/venv/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib
  Expected in:     <A5DBF409-59EB-37C4-ABE0-CF000BC95F69> /Users/vanshikaarora/Desktop/Mask-rcnn-from-scratch-personal/venv/lib/python3.11/site-packages/tensorflow/python/_pywrap_tensorflow_internal.so

In [ ]:
############################################################
#  Utility Functions
############################################################


def log(text, array=None):
    """Prints a text message. And, optionally, if a Numpy array is provided it
    prints it's shape, min, and max values.
     """
    """It is basically
    DEBUGGING UTILITY - Saves 45+ lines of manual print code.
    
    Instead of writing 5-6 print statements per tensor (shape, min, max, dtype),
    this does it all in 1 line. Critical for monitoring 10+ tensors:
    - Input images, 5 FPN feature maps, thousands of anchors, proposal boxes,
    - Class predictions, bbox refinements, masks, and more.
    
    Example: log("Feature map", tensor)  # 1 line vs 6 lines manually
    """
    if array is not None:
        text = text.ljust(25)
        text += ("shape: {:20}  ".format(str(array.shape)))
        if array.size:
            text += ("min: {:10.5f}  max: {:10.5f}".format(array.min(),array.max()))
        else:
            text += ("min: {:10}  max: {:10}".format("",""))
        text += "  {}".format(array.dtype)
    print(text)

class BatchNorm(KL.BatchNormalization):
    """Extends the Keras BatchNormalization class to allow a central place
    to make changes if needed.

    Batch normalization has a negative effect on training if batches are small
    so this layer is often frozen (via setting in Config class) and functions
    as linear layer.
    """
    """
    CALCULATES FEATURE MAP SIZES at each backbone stage (C2 through C6).
    
    WHY NEEDED: Different FPN levels (P2-P6) have different resolutions.
    Each stride (4,8,16,32,64) tells us how much the image shrinks.
    
    EXAMPLE: 1024x1024 image with stride 4 -> 256x256 feature map.
    These sizes determine where to place anchors and which ROIs go to which pyramid level.
    
    RETURNS: Array of [height, width] for each stage: [[256,256], [128,128], [64,64], [32,32], [16,16]]
    """
    def call(self, inputs, training=None):
        """
        Note about training values:
            None: Train BN layers. This is the normal mode
            False: Freeze BN layers. Good when batch size is small
            True: (don't use). Set layer in training mode even when making inferences
        """
        return super(self.__class__, self).call(inputs, training=training)
    

def compute_backbone_shapes(config, image_shape):
    """Computes the width and height of each stage of the backbone network.

    Returns:
        [N, (height, width)]. Where N is the number of stages
    """
    """
    PURPOSE: Calculate output dimensions of each ResNet stage (C2-C6).
    
    HOW IT WORKS: For each stride (4,8,16,32,64), compute: ceil(image_size / stride)
    - Stride 4  -> Stage C2 (256x256 for 1024 input)
    - Stride 8  -> Stage C3 (128x128)
    - Stride 16 -> Stage C4 (64x64)
    - Stride 32 -> Stage C5 (32x32)
    - Stride 64 -> Stage C6 (16x16) - RPN only
    
    WHY NEEDED: 
    - Determines anchor grid positions
    - Assigns ROIs to correct FPN levels
    - Validates image size compatibility
    
    RETURNS: np.array([[H1,W1], [H2,W2], [H3,W3], [H4,W4], [H5,W5]])
    """
    if callable(config.BACKBONE):
        return config.COMPUTE_BACKBONE_SHAPE(image_shape)

    # Currently supports ResNet only
    assert config.BACKBONE in ["resnet50", "resnet101"]
    return np.array(
        [[int(math.ceil(image_shape[0] / stride)),
            int(math.ceil(image_shape[1] / stride))]
            for stride in config.BACKBONE_STRIDES])

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    HOW CONV + IDENTITY BLOCKS WORK TOGETHER                        │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  INPUT IMAGE (224x224x3)                                                           │
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 1: Conv7x7 + MaxPool → 56x56x64                                         ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 2: (NO SHAPE CHANGE - stride=1)                                         ││
│  │                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  CONV BLOCK (block='a')                                                  │  ││
│  │  │  Input: 56x56x64                                                         │  ││
│  │  │  Output: 56x56x256  ← Changes channels only (64→256)                    │  ││
│  │  │  ★ This is the "DIMENSION CHANGER"                                      │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='b')                                              │  ││
│  │  │  Input: 56x56x256                                                        │  ││
│  │  │  Output: 56x56x256  ← NO SHAPE CHANGE!                                  │  ││
│  │  │  ★ This is the "FEATURE ENHANCER"                                       │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='c')                                              │  ││
│  │  │  Input: 56x56x256                                                        │  ││
│  │  │  Output: 56x56x256  ← NO SHAPE CHANGE!                                  │  ││
│  │  │  ★ More feature refinement                                              │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │                                                                                 ││
│  │  RESULT: C2 = 56x56x256 (same shape after 3 blocks)                           ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 3: (SHAPE CHANGE - stride=2)                                            ││
│  │                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  CONV BLOCK (block='a')                                                  │  ││
│  │  │  Input: 56x56x256                                                        │  ││
│  │  │  Output: 28x28x512  ← DOWNSAMPLES + INCREASES CHANNELS!                 │  ││
│  │  │  ★ Conv Block changes shape again!                                      │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='b')                                              │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 28x28x512  ← NO CHANGE!                                        │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='c')                                              │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 28x28x512  ← NO CHANGE!                                        │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='d')                                              │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 28x28x512  ← NO CHANGE!                                        │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │                                                                                 ││
│  │  RESULT: C3 = 28x28x512 (changed shape once, refined 3 times)                 ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 4: (SHAPE CHANGE - stride=2)                                            ││
│  │                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  CONV BLOCK (block='a')                                                  │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 14x14x1024 ← DOWNSAMPLES + INCREASES CHANNELS!                 │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK × (5 for ResNet50, 22 for ResNet101)                    │  ││
│  │  │  Each: 14x14x1024 → 14x14x1024 (NO CHANGE)                             │  ││
│  │  │  ★ MANY identity blocks = DEEPER network!                               │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │                                                                                 ││
│  │  RESULT: C4 = 14x14x1024                                                       ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 5: (SHAPE CHANGE - stride=2)                                            ││
│  │                                                                                 ││
│  │  CONV BLOCK (block='a') + 2x IDENTITY BLOCK                                    ││
│  │  Input: 14x14x1024 → Output: 7x7x2048                                          ││
│  │                                                                                 ││
│  │  RESULT: C5 = 7x7x2048                                                          ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│                                                                                     │
│  FINAL OUTPUT: [C1, C2, C3, C4, C5]                                               │
│                                                                                     │
│  ★ EACH STAGE: 1 Conv Block (changes shape) + N Identity Blocks (refines)         │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
Why This Pattern Works
1. Conv Block = "Gateway to New Resolution"
Purpose: Change the scale

What it does: Downsamples image, increases channels

Why: To detect larger objects (more semantic features)

When: Only at the start of each stage

2. Identity Blocks = "Feature Learners at That Resolution"
Purpose: Learn complex features at this scale

What it does: Adds more layers, more capacity

Why: To learn increasingly complex patterns

When: Multiple times after the Conv Block

3. Together = "Hierarchical Feature Learning"
text
Stage 2: Learn fine details at 56x56
Stage 3: Learn medium features at 28x28
Stage 4: Learn large features at 14x14
Stage 5: Learn semantic concepts at 7x7


The number of identity blocks in each stage is fixed and known from the ResNet paper:

Stage	ResNet50	ResNet101	ResNet152
Stage 2	2 identity blocks	2 identity blocks	3 identity blocks
Stage 3	3 identity blocks	3 identity blocks	8 identity blocks
Stage 4	5 identity blocks	22 identity blocks	36 identity blocks
Stage 5	2 identity blocks	2 identity blocks	3 identity blocks
Total Identity Blocks:

ResNet50: 2 + 3 + 5 + 2 = 12 identity blocks

ResNet101: 2 + 3 + 22 + 2 = 29 identity blocks

ResNet152: 3 + 8 + 36 + 3 = 50 identity blocks



You don't "use ResNet separately" - you use the complete trained Mask R-CNN model!

text
Trained Mask R-CNN Model:
┌─────────────────────────────────────────────────────────────┐
│  ┌─────────────────────────────────────────────────────┐   │
│  │  RESNET LAYERS (Trained as part of Mask R-CNN)     │   │
│  └─────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  FPN LAYERS (Trained as part of Mask R-CNN)        │   │
│  └─────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  RPN LAYERS (Trained as part of Mask R-CNN)        │   │
│  └─────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  HEADS (Trained as part of Mask R-CNN)             │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
It's ONE model, trained ONCE, with ALL layers updated together! 🎯



In [ ]:
############################################################
#  Resnet Graph
############################################################
def identity_block(input_tensor, kernel_size, filters, stage, block,
                   use_bias=True, train_bn=True):
    """The identity_block is the block that has no conv layer at shortcut
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of middle conv layer at main path
        filters: list of integers, the nb_filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
        block: 'a','b'..., current block label, used for generating layer names
        use_bias: Boolean. To use or not use a bias in conv layers.
        train_bn: Boolean. Train or freeze Batch Norm layers
    """
    nb_filter1, nb_filter2, nb_filter3 = filters
    conv_name_base='res'+str(stage)+block+'_branch'
    bn_name_base='bn'+str(stage)+block+'_branch'
    x=KL.Conv2D(nb_filter1,(1,1),name=conv_name_base+'2a',use_bias=use_bias)(input_tensor)
    x=BatchNorm(name=bn_name_base+'2a')(x,training=train_bn)
    x=KL.Activation('relu')(x)
    x = KL.Conv2D(nb_filter2, (kernel_size, kernel_size), padding='same',
                  name=conv_name_base + '2b', use_bias=use_bias)(x)
    x=BatchNorm(name=conv_name_base+'2b')(x,training=train_bn)
    x=KL.Activation('relu')(x)
    x = KL.Conv2D(nb_filter3, (1, 1), name=conv_name_base + '2c',
                  use_bias=use_bias)(x)
    x = BatchNorm(name=bn_name_base + '2c')(x, training=train_bn)
    x=KL.add()([x,input_tensor])#This single line does the skip connection!
## Element-wise addition:
# output[i] = x[i] + input_tensor[i]
# output = F(input) + input
      # = processed_features + original_input

    x = KL.Activation('relu', name='res' + str(stage) + block + '_out')(x)#Apply ReLU to the sum: output = ReLU(F(input) + input) its final activation
    #Why This is Called "Identity" Block
#The "Identity" refers to the shortcut path:
    return x
    """stage: 2, 3, 4, 5        # Which ResNet stage (C2, C3, C4, C5)
block: 'a', 'b', 'c'...   # Which block within that stage (first, second, third...) Why They Matter - Layer Naming!
Every layer needs a UNIQUE name so TensorFlow/Keras can track them.

python
# Example: Stage 2, Block 'b'
stage = 2
block = 'b'

conv_name_base = 'res' + str(stage) + block + '_branch'
# = 'res2b_branch'

bn_name_base = 'bn' + str(stage) + block + '_branch'
# = 'bn2b_branch'

# Now each layer gets a unique name:
# conv_name_base + '2a' = 'res2b_branch2a'
# conv_name_base + '2b' = 'res2b_branch2b'
# conv_name_base + '2c' = 'res2b_branch2c'
# bn_name_base + '2a' = 'bn2b_branch2a'
# etc. """
    
"""                  WHY x DOESN'T LOSE VALUE                                        │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  Think of it like making a sandwich:                                              │
│                                                                                     │
│  input_tensor = "BREAD"  (Original ingredient, NEVER CHANGES)                     │
│                                                                                     │
│  Step 1: x = "BREAD" + "BUTTER"  (x becomes buttered bread)                       │
│  Step 2: x = "BUTTERED BREAD" + "CHEESE"  (x becomes cheesy bread)               │
│  Step 3: x = "CHEESY BREAD" + "HAM"  (x becomes ham sandwich)                    │
│  Step 4: x = "HAM SANDWICH" + "TOMATO"  (x becomes complete sandwich)            │
│                                                                                     │
│  ★ input_tensor is still "BREAD" (unchanged!)                                     │
│  ★ x has been overwritten 4 times, but that's okay!                               │
│                                                                                     │
│  At the end:                                                                       │
│  x = complete sandwich                                                             │
│  input_tensor = bread (still there!)"""



"""Block Type	Main Path (Layers)	Shortcut Path
Identity Block	✅ Has layers (1x1 → 3x3 → 1x1)	❌ No layers (identity, direct pass)
Conv Block	✅ Has layers (1x1 → 3x3 → 1x1)	✅ Has layers (1x1 convolution)
What Each Block Actually Contributes
Conv Block Contribution: "Dimension Transformer"
python
Input:  [batch, 56, 56, 256]   # High resolution, 256 channels
        ↓
Conv Block
        ↓
Output: [batch, 28, 28, 512]   # Half resolution, double channels
What it contributes:

Downsamples spatial size (56×56 → 28×28)

Increases channels (256 → 512)

Reduces computation for deeper layers

Creates more semantic features (larger receptive field)

Identity Block Contribution: "Feature Enhancer"
python
Input:  [batch, 28, 28, 512]   # Fixed size, fixed channels
        ↓
Identity Block (3x in a row)
        ↓
Output: [batch, 28, 28, 512]   # SAME size, SAME channels
What it contributes:

Learns richer features at the same resolution

No shape change - just improves feature quality

Skip connections help gradients flow

Each block refines the features slightly more
so basically resnet architecture is lik it has two blocks identifty and conv block both do exact
 same thing like same layers goes through both but identituy has skip connectiuon to give output in a diff 
way and conv block doe siut the normal way that all other cn architecture 
# Both blocks have THESE EXACT SAME layers:
Conv1x1 → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN
in identity x = KL.Add()([x, input_tensor])  # Direct addition
Conv block does it the normal way"
YES! Conv block adds via a learned shortcut:

python
shortcut = KL.Conv2D(...)(input_tensor)  # Learned transformation
x = KL.Add()([x, shortcut])


Now todether how they are used
he Only Difference Summarized
Aspect	Identity Block	Conv Block
Main Path	✅ SAME	✅ SAME
Layers	✅ SAME	✅ SAME
Shortcut	Direct (no layers)	Learned (has Conv1x1)
When used	When shapes match	When shapes change

 Both blocks together create the feature hierarchy!


 
 Conv Block (changes shape) → Identity Blocks (refine features) → Conv Block (changes shape again) → Identity Blocks (refine again) → ...
 The Pattern: "One Conv Block + Many Identity Blocks"
In Each Stage:
text
STAGE 2: [CONV BLOCK] + [IDENTITY] + [IDENTITY]
STAGE 3: [CONV BLOCK] + [IDENTITY] + [IDENTITY] + [IDENTITY]
STAGE 4: [CONV BLOCK] + [IDENTITY] × (5 or 22)
STAGE 5: [CONV BLOCK] + [IDENTITY] + [IDENTITY]
Why This Pattern?
Block Type	When It's Used	How Many Times
Conv Block	ONCE at the start of each stage	1 per stage
Identity Block	MANY TIMES after the Conv Block	2-22 per stage
"""

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    BOTH BLOCKS HAVE SHORTCUTS!                                     │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  Identity Block:                                                                    │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  shortcut = input_tensor (DIRECT, NO LAYERS!)                                  ││
│  │  x = Add()([x, shortcut])                                                      ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Conv Block:                                                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  shortcut = Conv2D(input_tensor) (WITH LAYERS!)                                ││
│  │  x = Add()([x, shortcut])                                                      ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  ★ BOTH have shortcuts! The DIFFERENCE is what's IN the shortcut                  │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

# Identity Block shortcut:
# shortcut = input_tensor  (DIRECT, no processing!)
x = KL.Add()([x, input_tensor])
#            ↑
#            This is the shortcut - just passes input through

# Visual:
input_tensor ──────────────────────────────────────┐
    │                                               │
    ▼                                               │
Conv1x1 → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN
    │                                               │
    └───────────────┬───────────────────────────────┘
                    ▼
                Add() ←──┐
                    │    │  shortcut = input_tensor
                    ▼    │  (NO LAYERS, just identity)
                ReLU     │
                    │    │
                    ▼    │
                output ──┘

# Conv Block shortcut:
shortcut = KL.Conv2D(nb_filter3, (1, 1), strides=strides)(input_tensor)
#          ↑
#          This IS the shortcut - has a Conv2D layer!
shortcut = BatchNorm()(shortcut)
x = KL.Add()([x, shortcut])

# Visual:
input_tensor ──────────────────────────────────────┐
    │                                               │
    ▼                                               │
Conv1x1(stride) → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN
    │                                               │
    │                                               │
    └───────────────┬───────────▼───────────────────┘
                    │       Conv1x1(stride) → BN
                    │       ↑
                    │       shortcut = Conv2D(input_tensor)
                    │       (HAS LAYERS!)
                    │
                    └───────┬───────────────┘
                            ▼
                        Add()
                            │
                            ▼
                        ReLU
                            │
                            ▼
                        output
                        okay so just the diff is the identitu block does not have  has conv layer in shirtcut and whle conv block does have it

In [ ]:
def conv_block(input_tensor, kernel_size, filters, stage, block,
               strides=(2, 2), use_bias=True, train_bn=True):
    """conv_block is the block that has a conv layer at shortcut
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of middle conv layer at main path
        filters: list of integers, the nb_filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
        block: 'a','b'..., current block label, used for generating layer names
        use_bias: Boolean. To use or not use a bias in conv layers.
        train_bn: Boolean. Train or freeze Batch Norm layers
    Note that from stage 3, the first conv layer at main path is with subsample=(2,2)
    And the shortcut should have subsample=(2,2) as well
    """
    nb_filter1, nb_filter2, nb_filter3 = filters
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    x = KL.Conv2D(nb_filter1, (1, 1), strides=strides,
                  name=conv_name_base + '2a', use_bias=use_bias)(input_tensor)
    x = BatchNorm(name=bn_name_base + '2a')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter2, (kernel_size, kernel_size), padding='same',
                  name=conv_name_base + '2b', use_bias=use_bias)(x)
    x = BatchNorm(name=bn_name_base + '2b')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter3, (1, 1), name=conv_name_base +
                  '2c', use_bias=use_bias)(x)
    x = BatchNorm(name=bn_name_base + '2c')(x, training=train_bn)
    shortcut=KL.Conv2D(nb_filter3, (1, 1), strides=strides,
                         name=conv_name_base + '1', use_bias=use_bias)(input_tensor)
    shortcut = BatchNorm(name=bn_name_base + '1')(shortcut, training=train_bn)
    """The +1 in the shortcut name ('branch1') identifies it as the SHORTCUT PATH, while 2a, 2b, 2c are the MAIN PATH layers.

1 = Branch 1 (shortcut) - the skip connection path

2a, 2b, 2c = Branch 2 (main path) - the conv layers

Why? To give each layer a UNIQUE name so TensorFlow can track them separately. The shortcut needs its own conv layer in Conv Block to match dimensions, so it gets labeled branch1.

"""
    x = KL.Add()([x, shortcut])
    x = KL.Activation('relu', name='res' + str(stage) + block + '_out')(x)
    return x

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    WHY THIS STRUCTURE                                              │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  RESNET50:                                                                         │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Stage 1: Conv7x7 + MaxPool                                                    ││
│  │  Stage 2: Conv + 2x Identity → C2                                              ││
│  │  Stage 3: Conv + 3x Identity → C3                                              ││
│  │  Stage 4: Conv + 5x Identity → C4  ← block_count = 5                          ││
│  │  Stage 5: Conv + 2x Identity → C5                                              ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  RESNET101:                                                                        │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Stage 1: Conv7x7 + MaxPool                                                    ││
│  │  Stage 2: Conv + 2x Identity → C2                                              ││
│  │  Stage 3: Conv + 3x Identity → C3                                              ││
│  │  Stage 4: Conv + 22x Identity → C4  ← block_count = 22 (DEEPER!)              ││
│  │  Stage 5: Conv + 2x Identity → C5                                              ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  ★ SAME BLOCKS, JUST MORE IDENTITY BLOCKS IN STAGE 4!                              │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

In [ ]:
def resnet_graph(input_image, architecture, stage5=False, train_bn=True):
    """Build a ResNet graph.
        architecture: Can be resnet50 or resnet101
        stage5: Boolean. If False, stage5 of the network is not created
        train_bn: Boolean. Train or freeze Batch Norm layers
    """

    """ResNet50	50 layers	Shallower (faster, less memory)
ResNet101	101 layers	Deeper (more accurate, slower)
They use the SAME conv and identity blocks - just different NUMBER of identity blocks in Stage 4!
block_count = {"resnet50": 5, "resnet101": 22}[architecture]
This decides how many identity blocks to put in Stage 4!

C4 is the OUTPUT of Stage 4. The loop keeps updating x, and after ALL identity blocks, C4 gets the final valu

stage5 value	What happens	Why
True	Create Stage 5 → C5	For deeper networks (full ResNet)
False	Skip Stage 5 → C5 = None	For smaller networks (save memory)

Stage 1 is just the "entry layer" - it's NOT a ResNet stage with blocks. It's the initial convolution + pooling that happens BEFORE the actual ResNet stages begin
"""
    assert architecture in ["resnet50", "resnet101"]
    # Stage 1
    x = KL.ZeroPadding2D((3, 3))(input_image)
    x = KL.Conv2D(64, (7, 7), strides=(2, 2), name='conv1', use_bias=True)(x)
    x = BatchNorm(name='bn_conv1')(x, training=train_bn)
    x = KL.Activation('relu')(x)
    C1 = x = KL.MaxPooling2D((3, 3), strides=(2, 2), padding="same")(x)
    # Stage 2
    x = conv_block(x, 3, [64, 64, 256], stage=2, block='a', strides=(1, 1), train_bn=train_bn)
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='b', train_bn=train_bn)
    C2 = x = identity_block(x, 3, [64, 64, 256], stage=2, block='c', train_bn=train_bn)
     # Stage 3
    x = conv_block(x, 3, [128, 128, 512], stage=3, block='a', train_bn=train_bn)
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='b', train_bn=train_bn)
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='c', train_bn=train_bn)
    C3 = x = identity_block(x, 3, [128, 128, 512], stage=3, block='d', train_bn=train_bn)
    # Stage 4
    x = conv_block(x, 3, [256, 256, 1024], stage=4, block='a', train_bn=train_bn)
    block_count = {"resnet50": 5, "resnet101": 22}[architecture]
    for i in range(block_count):
        x = identity_block(x, 3, [256, 256, 1024], stage=4, block=chr(98 + i), train_bn=train_bn)
    C4 = x
    # Stage 5
    if stage5:
        x = conv_block(x, 3, [512, 512, 2048], stage=5, block='a', train_bn=train_bn)
        x = identity_block(x, 3, [512, 512, 2048], stage=5, block='b', train_bn=train_bn)
        C5 = x = identity_block(x, 3, [512, 512, 2048], stage=5, block='c', train_bn=train_bn)
    else:
        C5 = None
    return [C1, C2, C3, C4, C5]


┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    FILE ORDER vs EXECUTION ORDER                                    │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  FILE ORDER (Top to Bottom):                                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  1. Utility functions (log, BatchNorm, compute_backbone_shapes)                ││
│  │  2. ResNet blocks (identity_block, conv_block, resnet_graph)  ← Defined here   ││
│  │  3. Proposal Layer                                                             ││
│  │  4. ROI Align Layer                                                           ││
│  │  5. Detection Target Layer                                                    ││
│  │  6. Detection Layer                                                           ││
│  │  7. RPN (rpn_graph, build_rpn_model)                                         ││
│  │  8. FPN HEADS (fpn_classifier_graph, build_fpn_mask_graph)  ← Defined here   ││
│  │  9. Loss Functions                                                            ││
│  │  10. Data Generator                                                           ││
│  │  11. MaskRCNN Class                                                           ││
│  │      └─── build() method  ← FPN IS BUILT HERE!                              ││
│  │      └─── train() method                                                      ││
│  │      └─── detect() method                                                     ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  EXECUTION ORDER (When image flows through):                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 1: resnet_graph()  ← Called from build()                                ││
│  │  STEP 2: FPN (P2-P6)     ← Called from build()                                ││
│  │  STEP 3: RPN             ← Called from build()                                ││
│  │  STEP 4: ProposalLayer   ← Called from build()                                ││
│  │  STEP 5: ROI Align       ← Called from build()                                ││
│  │  STEP 6: fpn_classifier_graph()  ← Called from build()                       ││
│  │  STEP 7: build_fpn_mask_graph()  ← Called from build()                       ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

FPN is a network that creates feature maps at MULTIPLE SCALES so the model can detect objects of ALL sizes.

Without FPN: Only one feature map (C5) → good for large objects, bad for small ones
With FPN: Multiple feature maps (P2-P6) → good for ALL object sizes!
C5 (32x32) ──► P5 (32x32)  ← For LARGE objects
    ↓ Upsample
C4 (64x64) ──► P4 (64x64)  ← For MEDIUM-LARGE objects
    ↓ Upsample
C3 (128x128) ──► P3 (128x128) ← For MEDIUM objects
    ↓ Upsample
C2 (256x256) ──► P2 (256x256) ← For SMALL objects
. FPN Construction (The Pyramid Itself)
python
# This builds the actual pyramid - NOT a function, just code in build()
P5 = KL.Conv2D(256, (1, 1), name='fpn_c5p5')(C5)
P4 = KL.Add(...)([...])
P3 = KL.Add(...)([...])
P2 = KL.Add(...)([...])
P6 = KL.MaxPooling2D(...)(P5)
2. FPN Heads (The Functions You Posted)
python
# These USE the pyramid features - these ARE functions
fpn_classifier_graph()   # Uses P2-P5 to classify objects
build_fpn_mask_graph()   # Uses P2-P5 to make masks

def build(self, mode, config):
    # ============================================================
    # STEP 1: RESNET BACKBONE
    # ============================================================
    _, C2, C3, C4, C5 = resnet_graph(input_image, config.BACKBONE,
                                     stage5=True, train_bn=config.TRAIN_BN)
    # ↑ Output: C2, C3, C4, C5
    
    # ============================================================
    # STEP 2: FPN CONSTRUCTION (THE PYRAMID!)
    # ============================================================
    # THIS IS THE FPN ITSELF!
    P5 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c5p5')(C5)
    P4 = KL.Add(name="fpn_p4add")([
        KL.UpSampling2D(size=(2, 2), name="fpn_p5upsampled")(P5),
        KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c4p4')(C4)
    ])
    P3 = KL.Add(name="fpn_p3add")([
        KL.UpSampling2D(size=(2, 2), name="fpn_p4upsampled")(P4),
        KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c3p3')(C3)
    ])
    P2 = KL.Add(name="fpn_p2add")([
        KL.UpSampling2D(size=(2, 2), name="fpn_p3upsampled")(P3),
        KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c2p2')(C2)
    ])
    
    # Final 3x3 conv on each P layer
    P2 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p2")(P2)
    P3 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p3")(P3)
    P4 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p4")(P4)
    P5 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p5")(P5)
    
    # P6 for RPN
    P6 = KL.MaxPooling2D(pool_size=(1, 1), strides=2, name="fpn_p6")(P5)
    
    # ↑↑↑ FPN CONSTRUCTION COMPLETE: P2, P3, P4, P5, P6
    
    # ============================================================
    # STEP 3: RPN (Uses P2-P6)
    # ============================================================
    rpn_feature_maps = [P2, P3, P4, P5, P6]  # ← FPN features go to RPN!
    rpn = build_rpn_model(...)
    for p in rpn_feature_maps:
        layer_outputs.append(rpn([p]))
    rpn_class_logits, rpn_class, rpn_bbox = outputs
    
    # ============================================================
    # STEP 4: PROPOSAL LAYER
    # ============================================================
    rpn_rois = ProposalLayer(...)([rpn_class, rpn_bbox, anchors])
    
    # ============================================================
    # STEP 5: DETECTION TARGET LAYER (Training only)
    # ============================================================
    rois, target_class_ids, target_bbox, target_mask = DetectionTargetLayer(...)(
        [target_rois, input_gt_class_ids, gt_boxes, input_gt_masks])
    
    # ============================================================
    # STEP 6: FPN CLASSIFIER HEAD ← YOUR FIRST FUNCTION!
    # ============================================================
    mrcnn_feature_maps = [P2, P3, P4, P5]  # ← FPN features for heads!
    
    mrcnn_class_logits, mrcnn_class, mrcnn_bbox = fpn_classifier_graph(
        rois,                    # ← Proposals
        mrcnn_feature_maps,      # ← P2, P3, P4, P5 (FROM FPN!)
        input_image_meta,
        config.POOL_SIZE,
        config.NUM_CLASSES,
        train_bn=config.TRAIN_BN,
        fc_layers_size=config.FPN_CLASSIF_FC_LAYERS_SIZE
    )
    # ↑↑↑ fpn_classifier_graph() EXECUTED HERE!
    
    # ============================================================
    # STEP 7: FPN MASK HEAD ← YOUR SECOND FUNCTION!
    # ============================================================
    mrcnn_mask = build_fpn_mask_graph(
        rois,                    # ← Proposals
        mrcnn_feature_maps,      # ← P2, P3, P4, P5 (FROM FPN!)
        input_image_meta,
        config.MASK_POOL_SIZE,
        config.NUM_CLASSES,
        train_bn=config.TRAIN_BN
    )
    # ↑↑↑ build_fpn_mask_graph() EXECUTED HERE!

Final Summary: What FPN Does and Where It Fits
FPN (Feature Pyramid Network) is the bridge between ResNet's single-scale features and Mask R-CNN's multi-scale detection needs. It takes the four feature maps from ResNet—C2 (256×256, fine details), C3 (128×128, object parts), C4 (64×64, whole objects), and C5 (32×32, semantic context)—and transforms them into a five-level pyramid: P2 (256×256), P3 (128×128), P4 (64×64), P5 (32×32), and P6 (16×16). It does this by first reducing all channels to 256 via 1×1 convolutions (lateral connections), then repeatedly upsampling the deeper, more semantic features and adding them to the shallower, more detailed features (top-down pathway), and finally smoothing each level with a 3×3 convolution. The result is a set of feature maps that all have the same channel depth (256) but different spatial resolutions, allowing the network to detect objects at every scale—P2 for tiny objects, P3 for medium, P4 for large, P5 for very large, and P6 (created by max-pooling P5) for the largest. These five maps are then passed directly to the RPN, which applies the same shared network to each level to generate region proposals (candidate boxes) across all scales, ensuring that no object—whether small or large—is missed. In essence, FPN gives RPN the ability to "see" objects of all sizes by providing a multi-scale feature pyramid built from ResNet's hierarchical features.

In [ ]:
############################################################
#  Region Proposal Network (RPN)
############################################################
def rpn_graph(feature_map, anchors_per_location, anchor_stride):
    """Builds the computation graph of Region Proposal Network.

    feature_map: backbone features [batch, height, width, depth]
    anchors_per_location: number of anchors per pixel in the feature map
    anchor_stride: Controls the density of anchors. Typically 1 (anchors for
                   every pixel in the feature map), or 2 (every other pixel).

    Returns:
        rpn_class_logits: [batch, H * W * anchors_per_location, 2] Anchor classifier logits (before softmax)
        rpn_probs: [batch, H * W * anchors_per_location, 2] Anchor classifier probabilities.
        rpn_bbox: [batch, H * W * anchors_per_location, (dy, dx, log(dh), log(dw))] Deltas to be
                  applied to anchors.
    """
    # TODO: check if stride of 2 causes alignment issues if the feature map
    # is not even.
    # Shared convolutional base of the RPN
    #Applies 3×3 convolution with 512 filters
    #Why "shared"? Both heads (classification and bbox) use these features.

    shared = KL.Conv2D(512, (3, 3), padding='same', activation='relu',
                       strides=anchor_stride,
                       name='rpn_conv_shared')(feature_map)
    # Anchor Score. [batch, height, width, anchors per location * 2].
    #Step 2: Classification Head (Is it FG or BG?) For each anchor: predicts 2 values (FG score, BG score)
    x = KL.Conv2D(2 * anchors_per_location, (1, 1), padding='valid',
                  activation='linear', name='rpn_class_raw')(shared)
    # Reshape to [batch, anchors, 2] Reshapes from [batch, H, W, 2*A] to [batch, H*W*A, 2]
    rpn_class_logits = KL.Lambda(
        lambda t: tf.reshape(t, [tf.shape(input=t)[0], -1, 2]))(x)
    # Softmax on last dimension of BG/FG. Applies softmax to get probabilities probability of background, probability of foreground (object)
    rpn_probs = KL.Activation(
        "softmax", name="rpn_class_xxx")(rpn_class_logits)
    #BBox Regression Head (How to adjust the box)
    # Bounding box refinement. [batch, H, W, anchors per location * depth]
    # where depth is [x, y, log(w), log(h)]
    x = KL.Conv2D(anchors_per_location * 4, (1, 1), padding="valid",
                  activation='linear', name='rpn_bbox_pred')(shared)
#     dy = how much to shift center_y (normalized by height)
# dx = how much to shift center_x (normalized by width)
# log(dh) = how much to change height (log scale)
# log(dw) = how much to change width (log scale)
    # Reshape to [batch, anchors, 4]
    """Example:Anchor: [100, 50, 200, 150]  (y1, x1, y2, x2)
Deltas: [0.1, -0.05, 0.2, 0.15]

Apply deltas:
center_y = 150 + 0.1*100 = 160
center_x = 100 + (-0.05)*100 = 95
height = 100 * exp(0.2) = 122
width = 100 * exp(0.15) = 116

Refined box: [160-61, 95-58, 160+61, 95+58] = [99, 37, 221, 153]"""
    rpn_bbox = KL.Lambda(lambda t: tf.reshape(t, [tf.shape(input=t)[0], -1, 4]))(x)
#Reshapes to [batch, H*W*A, 4]Each anchor has 4 deltas
    return [rpn_class_logits, rpn_probs, rpn_bbox]

""" What rpn_graph() Returns (Raw Outputs)
rpn_class_logits: [batch, total_anchors, 2] - Classification scores (before softmax)

rpn_probs: [batch, total_anchors, 2] - Classification probabilities (after softmax)

rpn_bbox: [batch, total_anchors, 4] - BBox deltas for each anchor

build_rpn_model() wraps rpn_graph() into a reusable Keras Model so the SAME RPN can be applied to multiple FPN levels with SHARED weights.
# Would need to manually create RPN for each level:
rpn_class_logits_p2, rpn_probs_p2, rpn_bbox_p2 = rpn_graph(P2, ...)
rpn_class_logits_p3, rpn_probs_p3, rpn_bbox_p3 = rpn_graph(P3, ...)
rpn_class_logits_p4, rpn_probs_p4, rpn_bbox_p4 = rpn_graph(P4, ...)
rpn_class_logits_p5, rpn_probs_p5, rpn_bbox_p5 = rpn_graph(P5, ...)
rpn_class_logits_p6, rpn_probs_p6, rpn_bbox_p6 = rpn_graph(P6, ...)

# Problem: Each would have SEPARATE weights! (not shared)"""
def build_rpn_model(anchor_stride, anchors_per_location, depth):
    """Builds a Keras model of the Region Proposal Network.
    It wraps the RPN graph so it can be used multiple times with shared
    weights.

    anchors_per_location: number of anchors per pixel in the feature map
    anchor_stride: Controls the density of anchors. Typically 1 (anchors for
                   every pixel in the feature map), or 2 (every other pixel).
    depth: Depth of the backbone feature map.

    Returns a Keras Model object. The model outputs, when called, are:
    rpn_class_logits: [batch, H * W * anchors_per_location, 2] Anchor classifier logits (before softmax)
    rpn_probs: [batch, H * W * anchors_per_location, 2] Anchor classifier probabilities.
    rpn_bbox: [batch, H * W * anchors_per_location, (dy, dx, log(dh), log(dw))] Deltas to be
                applied to anchors.
    """
    input_feature_map = KL.Input(shape=[None, None, depth],
                                 name="input_rpn_feature_map")
    outputs = rpn_graph(input_feature_map, anchors_per_location, anchor_stride)
    return KM.Model([input_feature_map], outputs, name="rpn_model")

    

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    RPN: CLASSIFICATION + BBOX HEADS                                │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  INPUT: Feature Map [batch, H, W, depth]                                           │
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  SHARED CONV: KL.Conv2D(512, (3,3), activation='relu')                         ││
│  │  Output: [batch, H, W, 512]                                                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                               │
│                    ┌───────────────┴───────────────┐                              │
│                    │                               │                              │
│                    ▼                               ▼                              │
│  ┌─────────────────────────────────┐  ┌──────────────────────────────────────────┐│
│  │  CLASSIFICATION HEAD            │  │  BBOX REGRESSION HEAD                    ││
│  │  ┌───────────────────────────┐  │  │  ┌──────────────────────────────────────┐││
│  │  │  KL.Conv2D(2*A, (1,1))    │  │  │  │  KL.Conv2D(4*A, (1,1))              │││
│  │  │  activation='linear'      │  │  │  │  activation='linear'                │││
│  │  │  Output: [batch,H,W,2A]   │  │  │  │  Output: [batch,H,W,4A]             │││
│  │  └───────────────────────────┘  │  │  └──────────────────────────────────────┘││
│  │              │                   │  │              │                          ││
│  │              ▼                   │  │              ▼                          ││
│  │  ┌───────────────────────────┐  │  │  ┌──────────────────────────────────────┐││
│  │  │  Reshape:                  │  │  │  │  Reshape:                           │││
│  │  │  [batch, H*W*A, 2]        │  │  │  │  [batch, H*W*A, 4]                 │││
│  │  │  rpn_class_logits          │  │  │  │  rpn_bbox                          │││
│  │  └───────────────────────────┘  │  │  └──────────────────────────────────────┘││
│  │              │                   │  │                                          ││
│  │              ▼                   │  │                                          ││
│  │  ┌───────────────────────────┐  │  │                                          ││
│  │  │  Softmax                   │  │  │                                          ││
│  │  │  rpn_probs                 │  │  │                                          ││
│  │  └───────────────────────────┘  │  │                                          ││
│  └─────────────────────────────────┘  └──────────────────────────────────────────┘│
│                    │                               │                              │
│                    └───────────────┬───────────────┘                              │
│                                    ▼                                               │
│  OUTPUT:                                                                          │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  rpn_class_logits: [batch, H*W*A, 2]  ← Classification (before softmax)       ││
│  │  rpn_probs:        [batch, H*W*A, 2]  ← Classification (after softmax)        ││
│  │  rpn_bbox:         [batch, H*W*A, 4]  ← BBox regression                       ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

How rpn being used in main mask block
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    ONE RPN MODEL, MANY FEATURE MAPS                                │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  rpn = Keras Model (created once)                                                 │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Input: feature_map [batch, H, W, 256]                                        ││
│  │  ├── Conv2D(512, 3x3)  ← weights w1, w2, w3 (SHARED!)                        ││
│  │  ├── Conv2D(2A, 1x1)   ← weights w4, w5 (SHARED!)                            ││
│  │  └── Conv2D(4A, 1x1)   ← weights w6, w7 (SHARED!)                            ││
│  │  Output: [class_logits, probs, bbox]                                          ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                               │
│        ┌───────────────────────────┼───────────────────────────┐                  │
│        │                           │                           │                  │
│        ▼                           ▼                           ▼                  │
│  ┌───────────┐              ┌───────────┐              ┌───────────┐             │
│  │    P2     │              │    P3     │              │    P4     │             │
│  │ 256x256   │              │ 128x128   │              │ 64x64     │             │
│  └─────┬─────┘              └─────┬─────┘              └─────┬─────┘             │
│        │                           │                           │                  │
│        ▼                           ▼                           ▼                  │
│  ┌───────────┐              ┌───────────┐              ┌───────────┐             │
│  │ rpn(P2)   │              │ rpn(P3)   │              │ rpn(P4)   │             │
│  │ Uses w1-w7│              │ Uses w1-w7│              │ Uses w1-w7│             │
│  │ (SHARED!) │              │ (SHARED!) │              │ (SHARED!) │             │
│  └─────┬─────┘              └─────┬─────┘              └─────┬─────┘             │
│        │                           │                           │                  │
│        ▼                           ▼                           ▼                  │
│  ┌───────────┐              ┌───────────┐              ┌───────────┐             │
│  │ [class,   │              │ [class,   │              │ [class,   │             │
│  │  probs,   │              │  probs,   │              │  probs,   │             │
│  │  bbox]    │              │  bbox]    │              │  bbox]    │             │
│  └───────────┘              └───────────┘              └───────────┘             │
│        │                           │                           │                  │
│        └───────────────────────────┼───────────────────────────┘                  │
│                                    ▼                                               │
│                    ┌─────────────────────────────────────────────────────────────────┐│
│                    │  Concatenate all levels:                                       ││
│                    │  rpn_class_logits: [batch, total_anchors, 2]                 ││
│                    │  rpn_probs:        [batch, total_anchors, 2]                 ││
│                    │  rpn_bbox:         [batch, total_anchors, 4]                 ││
│                    └─────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

okay so u mena when rpn is applied in build it takes all fpn outputs inside its keras model layer and creates outputs to all then those are feeded to porposal layer

What Goes INTO ProposalLayer
python
rpn_rois = ProposalLayer(...)([rpn_class, rpn_bbox, anchors])
#                              ↑           ↑          ↑
#                          Input 1    Input 2    Input 3
Input 1: rpn_class - Classification Scores
text
Shape: [batch, total_anchors, 2]
Contains: For each anchor → [BG_prob, FG_prob]
- FG_prob = probability this anchor contains an object
- BG_prob = probability this anchor is background
From ALL FPN levels combined! (P2, P3, P4, P5, P6)

Input 2: rpn_bbox - BBox Deltas
text
Shape: [batch, total_anchors, 4]
Contains: For each anchor → [dy, dx, log(dh), log(dw)]
- These tell how to refine the anchor box
From ALL FPN levels combined!

Input 3: anchors - Pre-defined Boxes
text
Shape: [batch, total_anchors, 4]
Contains: For each anchor → [y1, x1, y2, x2]
- The actual anchor boxes (in normalized coordinates)
Pre-generated for all FPN levels!



PROPOSAL LAYER - Simple Explanation

What It Does (In Simple Terms)

The Proposal Layer takes 785,000 rough anchor boxes and picks the best 2,000/1,000 boxes that actually contain objects. It's like having 785,000 guesses about where objects might be, keeping only the best 2,000 guesses, refining those guesses to fit objects better, and removing overlapping guesses so you don't have duplicates.

What Goes IN (Inputs)

Input: rpn_class, Shape: [batch, 785k, 2], What It Contains: FG/BG scores for every anchor from ALL FPN levels (P2-P6). Input: rpn_bbox, Shape: [batch, 785k, 4], What It Contains: 4 deltas [dy, dx, log(dh), log(dw)] for every anchor. Input: anchors, Shape: [batch, 785k, 4], What It Contains: Pre-defined anchor boxes [y1, x1, y2, x2].

What Happens INSIDE

785,000 anchors with scores and deltas go through the following steps: First, keep top 6,000 by FG score (best guesses). Second, refine each box using deltas to adjust position and size. Third, clip to image boundaries to keep boxes inside image. Fourth, apply NMS to remove overlapping boxes with IoU greater than 0.7. Fifth, keep top 2,000 for training or 1,000 for inference. Finally, output rpn_rois with shape [batch, 2000/1000, (y1, x1, y2, x2)].

What Comes OUT (Output)

The output is rpn_rois with shape [batch, proposal_count, (y1, x1, y2, x2)], where proposal_count is 2000 for training or 1000 for inference. Each box is in normalized coordinates ranging from 0 to 1.

Where It Goes (Output Destination)

The rpn_rois output goes to ROI ALIGN (PyramidROIAlign), which then extracts features for each proposal, passes them to Classification and Mask Heads, and finally produces the output with Classes, Refined Boxes, and Masks.

What Exactly Are Anchors?
Anchors are pre-defined boxes placed at every position on the image.
 Anchors are PRE-GENERATED (NOT predicted!)

Anchor Properties:
Fixed positions: Placed at regular intervals

Fixed sizes: From RPN_ANCHOR_SCALES (32, 64, 128, 256, 512)

Fixed ratios: From RPN_ANCHOR_RATIOS (0.5, 1:2, 2:1)

Total anchors: ~785,000 for a 1024x1024 image

NEXT


Question 1: What Are Deltas?
Deltas are the "instructions" on how to adjust an anchor box to better fit an object.

Simple Explanation:
Instead of predicting boxes directly, RPN predicts how much to change each anchor box. These changes are called "deltas."

The 4 Deltas:
python
deltas = [dy, dx, log(dh), log(dw)]
Delta	What It Does	How It Works
dy	Shift box up/down	center_y += dy * height
dx	Shift box left/right	center_x += dx * width
log(dh)	Change height	height *= exp(log(dh))
log(dw)	Change width	width *= exp(log(dw))
Example:
text
Anchor: [100, 50, 200, 150]  (100x100 box)
Deltas: [0.1, -0.05, 0.2, 0.15]

Meaning:
- Move center down by 10% of height
- Move center left by 5% of width
- Increase height by 22% (exp(0.2))
- Increase width by 16% (exp(0.15))

Result: [99, 37, 221, 153]  (refined box)
Question 2: How Are Deltas Predicted?
Deltas are the OUTPUT of RPN's BBox Regression Head!


What RPN Predicts:
text
For each anchor (785k anchors):
  - 2 scores: [BG_prob, FG_prob]  (Classification)
  - 4 deltas: [dy, dx, log(dh), log(dw)]  (BBox Regression)


   Anchors are PRE-GENERATED (NOT predicted!)

In [ ]:
############################################################
#  Proposal Layer
############################################################

def apply_box_deltas_graph(boxes, deltas):
    """Applies the given deltas to the given boxes.
    boxes: [N, (y1, x1, y2, x2)] boxes to update
    deltas: [N, (dy, dx, log(dh), log(dw))] refinements to apply
    """
    # Convert to y, x, h, w
    #It takes boxes defined by their corners [y1, x1, y2, x2] and converts them to [center_y, center_x, height, width].
#Step 1: Calculate Height and Width What this does: Subtracts top from bottom to get height, left from right to get width.
#Step 2: Calculate Center What this does: Adds half the height/width to the top/left corner to find the middle.
    height = boxes[:, 2] - boxes[:, 0]
    width = boxes[:, 3] - boxes[:, 1]
    center_y = boxes[:, 0] + 0.5 * height
    center_x = boxes[:, 1] + 0.5 * width
    # Apply deltas
    center_y += deltas[:, 0] * height
    center_x += deltas[:, 1] * width
    height *= tf.exp(deltas[:, 2])
    width *= tf.exp(deltas[:, 3])
    """What Each Delta Does:
Delta	Operation	Effect
deltas[:, 0] (dy)	center_y += dy * height	Move box vertically (up/down)
deltas[:, 1] (dx)	center_x += dx * width	Move box horizontally (left/right)
deltas[:, 2] (log(dh))	height *= exp(log(dh))	Change height (taller/shorter)
deltas[:, 3] (log(dw))	width *= exp(log(dw))	Change width (wider/narrower)

exp() ensures height and width are ALWAYS positive numbers!

STEP 1: CONVERT TO CENTER FORMAT
    [y1, x1, y2, x2] → [center_y, center_x, height, width]
    
STEP 2: APPLY DELTAS
    [center_y, center_x, height, width] + [dy, dx, log(dh), log(dw)]
    → [center_y', center_x', height', width']
    
STEP 3: CONVERT BACK TO CORNER FORMAT
    [center_y', center_x', height', width'] → [y1', x1', y2', x2']"""
    # Convert back to y1, x1, y2, x2
    y1 = center_y - 0.5 * height
    x1 = center_x - 0.5 * width
    y2 = y1 + height
    x2 = x1 + width
    result = tf.stack([y1, x1, y2, x2], axis=1, name="apply_box_deltas_out")
    return result


def clip_boxes_graph(boxes, window):
    """
    boxes: [N, (y1, x1, y2, x2)]
    window: [4] in the form y1, x1, y2, x2
    """
    # Split
    wy1, wx1, wy2, wx2 = tf.split(window, 4)
    y1, x1, y2, x2 = tf.split(boxes, 4, axis=1)
    # Clip
    y1 = tf.maximum(tf.minimum(y1, wy2), wy1)
    x1 = tf.maximum(tf.minimum(x1, wx2), wx1)
    y2 = tf.maximum(tf.minimum(y2, wy2), wy1)
    x2 = tf.maximum(tf.minimum(x2, wx2), wx1)
    clipped = tf.concat([y1, x1, y2, x2], axis=1, name="clipped_boxes")
    clipped.set_shape((clipped.shape[0], 4))
    return clipped
#Simply put: clip_boxes_graph() ensures that no box goes outside the image. It trims any part of a box that extends beyond the image boundaries, keeping all boxes valid for later processing.
#proposal_count is the number of region proposals the layer should output.Training	2000	More proposals = better training diversity
#Inference	1000	Fewer proposals = faster inference
"""What is nms_threshold?
nms_threshold is the IoU threshold for Non-Maximum Suppression.

IoU = Intersection over Union (how much two boxes overlap)

If IoU > nms_threshold, boxes are considered duplicates and removed

Threshold	Effect
0.7 (default)	Remove boxes that overlap >70% (balanced)
0.5 (lower)	Remove more boxes (fewer proposals)
0.9 (higher)	Keep more boxes (more proposals)"""
#get_config() saves the layer's configuration so it can be serialized (saved to disk) and reloaded later.

class ProposalLayer(KL.Layer):
    """Receives anchor scores and selects a subset to pass as proposals
    to the second stage. Filtering is done based on anchor scores and
    non-max suppression to remove overlaps. It also applies bounding
    box refinement deltas to anchors.

    Inputs:
        rpn_probs: [batch, num_anchors, (bg prob, fg prob)]
        rpn_bbox: [batch, num_anchors, (dy, dx, log(dh), log(dw))]
        anchors: [batch, num_anchors, (y1, x1, y2, x2)] anchors in normalized coordinates

    Returns:
        Proposals in normalized coordinates [batch, rois, (y1, x1, y2, x2)]
    """

    def __init__(self, proposal_count, nms_threshold, config=None, **kwargs):
        super(ProposalLayer, self).__init__(**kwargs)
        self.config = config
        self.proposal_count = proposal_count
        self.nms_threshold = nms_threshold

    def get_config(self):
        config = super(ProposalLayer, self).get_config()
        config["config"] = self.config.to_dict()
        config["proposal_count"] = self.proposal_count
        config["nms_threshold"] = self.nms_threshold
        return config

    def call(self, inputs):
        # Box Scores. Use the foreground class confidence. [Batch, num_rois, 1]Extract Foreground Scores
        #Why only FG? We only care about anchors that CONTAIN objects (foreground). Background anchors are useless to us.
        scores = inputs[0][:, :, 1]
        # Box deltas [batch, num_rois, 4]
        deltas = inputs[1]
        #What it does: Scales the deltas by a standard deviation factor.
        deltas = deltas * np.reshape(self.config.RPN_BBOX_STD_DEV, [1, 1, 4])
        # AnchorsWhat it does: Just stores the anchor boxes for later use.
        anchors = inputs[2]
  #This code is the Pre-NMS Selection step - it reduces ~785,000 anchors to the top 6,000 most promising ones.
        # Improve performance by trimming to top anchors by score
        # and doing the rest on the smaller subset.
        #Why scale? During training, RPN learns normalized deltas. Scaling brings them to the right magnitude.
        #Keep at most 6,000 anchors, or fewer if there aren't enough anchors."below line
        pre_nms_limit = tf.minimum(self.config.PRE_NMS_LIMIT, tf.shape(input=anchors)[1])
        #What it does: Sets how many anchors to keep before NMS.
        """Why 6000?

785k anchors is too many for NMS (slow!)

Keeping top 6,000 makes NMS much faster

The top 6,000 anchors are already the best candidates

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    TOP-K SELECTION                                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  All Anchors (785k):                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Index:  0     1     2     3     4     5     6     7     8     9    ...      ││
│  │  Score:  0.95  0.12  0.88  0.45  0.92  0.33  0.78  0.21  0.65  0.09  ...      ││
│  │           ↑                 ↑     ↑           ↑           ↑                     ││
│  │       highest             high   high        high        high                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Top 6,000 Scores:                                                                  │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  [0.95, 0.92, 0.88, 0.78, 0.65, 0.45, ...]  (6,000 highest scores)           ││
│  │   ↑     ↑     ↑     ↑     ↑     ↑                                              ││
│  │  [0,    4,    2,    6,    8,    3, ...]     ← These are the INDICES!         ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Result: ix = [0, 4, 2, 6, 8, 3, ...]  (indices of top 6,000 anchors)            │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘"""
#Top-K Selection (The Core!)
        ix = tf.nn.top_k(scores, pre_nms_limit, sorted=True,
                         name="top_anchors").indices
        #What it does: Finds the indices of the top pre_nms_limit anchors with highest FG scores.
        #What it does: Uses the indices (ix) to select only the top anchors from scores, deltas, and anchors.
        scores = utils.batch_slice([scores, ix], lambda x, y: tf.gather(x, y),
                                   self.config.IMAGES_PER_GPU)
        deltas = utils.batch_slice([deltas, ix], lambda x, y: tf.gather(x, y),
                                   self.config.IMAGES_PER_GPU)
        pre_nms_anchors = utils.batch_slice([anchors, ix], lambda a, x: tf.gather(a, x),
                                    self.config.IMAGES_PER_GPU,
                                    names=["pre_nms_anchors"])
        # Apply deltas to anchors to get refined anchors.
        # [batch, N, (y1, x1, y2, x2)]
        boxes = utils.batch_slice([pre_nms_anchors, deltas],
                                  lambda x, y: apply_box_deltas_graph(x, y),
                                  self.config.IMAGES_PER_GPU,
                                  names=["refined_anchors"])

        # Clip to image boundaries. Since we're in normalized coordinates,
        # clip to 0..1 range. [batch, N, (y1, x1, y2, x2)] Ensures all refined boxes stay within the image boundaries [0, 1].
        window = np.array([0, 0, 1, 1], dtype=np.float32)
        boxes = utils.batch_slice(boxes,
                                  lambda x: clip_boxes_graph(x, window),
                                  self.config.IMAGES_PER_GPU,
                                  names=["refined_anchors_clipped"])

        # Filter out small boxes
        # According to Xinlei Chen's paper, this reduces detection accuracy
        # for small objects, so we're skipping it.

        # Non-max suppression
        """What this does:

Takes 6,000 boxes and their scores

Removes boxes that overlap too much (IoU > nms_threshold)

Keeps at most proposal_count boxes
1. Sort boxes by score (highest first)
2. Pick highest score box → KEEP
3. Remove all boxes with IoU > 0.7 → DUPLICATES
4. Repeat until proposal_count boxes selected
Step 3: Pad to Exact Count
Part 4: Batch Processing
Part 5: Set Output Shape (Graph Mode)"""
        def nms(boxes, scores):
            indices = tf.image.non_max_suppression(
                boxes, scores, self.proposal_count,
                self.nms_threshold, name="rpn_non_max_suppression")
            proposals = tf.gather(boxes, indices)
            # Pad if needed
            padding = tf.maximum(self.proposal_count - tf.shape(input=proposals)[0], 0)
            proposals = tf.pad(tensor=proposals, paddings=[(0, padding), (0, 0)])
            return proposals
        proposals = utils.batch_slice([boxes, scores], nms,
                                      self.config.IMAGES_PER_GPU)

        if not context.executing_eagerly():
            # Infer the static output shape:
            out_shape = self.compute_output_shape(None)
            proposals.set_shape(out_shape)
        return proposals

    def compute_output_shape(self, input_shape):
        return None, self.proposal_count, 4
        


mportant Clarification: 0.3 and 0.7 are NOT in ProposalLayer!
These thresholds belong to RPN TRAINING (when anchors are labeled as positive/negative), NOT to ProposalLayer (which is inference).

OVERALL FULL PROPOSAL


┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    6,000 → 2,000/1,000 → ROI Align                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  STEP 1: Start with 785,000 anchors                                                │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 2: Keep top 6,000 by FG score                                                │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 3: Apply deltas → 6,000 refined boxes                                        │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 4: Clip to image → 6,000 clipped boxes                                       │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 5: NMS removes overlapping boxes!                                            │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  NMS (IoU > 0.7):                                                              ││
│  │  ┌───────────────────────────────────────────────────────────────────────────┐ ││
│  │  │  6,000 boxes → Remove overlaps → ~2,000 boxes (training)                 │ ││
│  │  │  6,000 boxes → Remove overlaps → ~1,000 boxes (inference)                │ ││
│  │  └───────────────────────────────────────────────────────────────────────────┘ ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  STEP 6: Pad to exact count                                                        │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 7: OUTPUT: rpn_rois [batch, 2000/1000, 4]                                   │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 8: ROI Align uses these 2,000/1,000 proposals!                               │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘